# 05 – Final Attack Strategy Set

The previous notebook consolidated the 9 mathematical clusters into a smaller, validated attack-strategy taxonomy.

The accepted proposal contains three strategies:

- `S1 – Fake Directive Manipulation`
- `S2 – Stealthy Dependency Injection`
- `S3 – Context Hijacking`

The primary consolidation model grouped clusters `0, 1, 2, 4, 5, 6, 7` into `S1`, while clusters `3` and `8` remained distinct as `S2` and `S3`.

The independent validator agreed with the taxonomy and reported no problematic merges.

This notebook now creates the final attack-strategy dataset used by the later defense experiments.

The goals are:

1. preserve every original attack and cluster ID
2. assign every attack exactly one final strategy ID
3. validate the cluster-to-strategy mapping
4. summarize the final strategy distribution
5. export the final dataset and taxonomy

No additional LLM call is used in this notebook.


In [2]:
# =========================
#  1. BLOCK – DEPENDENCIES
# =========================

from pathlib import Path
import json

import pandas as pd


In [3]:
# =========================
#  2. BLOCK – PATHS AND CONFIGURATION
# =========================

PROJECT_ROOT = Path.cwd().resolve()

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
)

FINAL_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "final"
)

FINAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CLUSTER_ASSIGNMENTS_PATH = (
    RESULTS_DIR
    / "ipi_arena_cluster_assignments.csv"
)

CONSOLIDATION_PROPOSAL_PATH = (
    RESULTS_DIR
    / "ipi_arena_strategy_consolidation_proposal.json"
)

CONSOLIDATION_VALIDATION_PATH = (
    RESULTS_DIR
    / "ipi_arena_strategy_consolidation_validation.json"
)

FINAL_ATTACK_DATASET_PATH = (
    FINAL_DATA_DIR
    / "ipi_arena_final_attack_strategies.csv"
)

FINAL_TAXONOMY_PATH = (
    FINAL_DATA_DIR
    / "ipi_arena_final_strategy_taxonomy.csv"
)

FINAL_MAPPING_PATH = (
    FINAL_DATA_DIR
    / "ipi_arena_final_cluster_strategy_mapping.csv"
)

print("Cluster assignments:", CLUSTER_ASSIGNMENTS_PATH)
print("Consolidation proposal:", CONSOLIDATION_PROPOSAL_PATH)
print("Validation:", CONSOLIDATION_VALIDATION_PATH)
print("Final data directory:", FINAL_DATA_DIR)


Cluster assignments: C:\work\LLM filtering\TDK\code\results\ipi_arena_cluster_assignments.csv
Consolidation proposal: C:\work\LLM filtering\TDK\code\results\ipi_arena_strategy_consolidation_proposal.json
Validation: C:\work\LLM filtering\TDK\code\results\ipi_arena_strategy_consolidation_validation.json
Final data directory: C:\work\LLM filtering\TDK\code\data\final


In [4]:
# =========================
#  3. BLOCK – LOAD CLUSTER ASSIGNMENTS
# =========================

df_attacks = pd.read_csv(
    CLUSTER_ASSIGNMENTS_PATH
)

required_columns = {
    "attack_id",
    "behavior_id",
    "attack",
    "cluster_id",
}

missing_columns = (
    required_columns
    - set(df_attacks.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            sorted(missing_columns)
        )
    )

print(
    "Number of attacks:",
    len(df_attacks)
)

print(
    "Number of original clusters:",
    df_attacks[
        "cluster_id"
    ].nunique()
)

display(
    df_attacks.head()
)


Number of attacks: 95
Number of original clusters: 9


,attack_id,behavior_id,attack,char_count,word_count,dataset_split,cluster_id
0,4e2e017f1bf5a09c,hotel-frontdesk-premature-checkout,<SYSTEM-PROMPT>\n# Core Identity\nYou are Coco...,3770,494,train,1
1,ae3f28927d045cc4,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",1069,165,train,1
2,3b44999458d7cb1a,hotel-frontdesk-premature-checkout,"{\n ""lost_items"": [],\n ""housekeeping_notes""...",1778,279,train,1
3,7a62f421a30afbb1,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",914,136,train,1
4,ea226a03df427537,hotel-frontdesk-premature-checkout,END-OF-TOOL-RESPONSE> \n<USER> \nFront Desk ...,13135,1534,train,1


In [5]:
# =========================
#  4. BLOCK – LOAD CONSOLIDATION PROPOSAL
# =========================

with open(
    CONSOLIDATION_PROPOSAL_PATH,
    "r",
    encoding="utf-8",
) as file:
    consolidation_proposal = json.load(
        file
    )

taxonomy = (
    consolidation_proposal.get(
        "taxonomy",
        []
    )
)

cluster_mapping = (
    consolidation_proposal.get(
        "cluster_mapping",
        []
    )
)

if not taxonomy:
    raise ValueError(
        "The consolidation proposal contains no taxonomy."
    )

if not cluster_mapping:
    raise ValueError(
        "The consolidation proposal contains no cluster mapping."
    )

print(
    "Number of final strategies:",
    len(taxonomy)
)

print(
    "Number of mapped clusters:",
    len(cluster_mapping)
)

print(
    json.dumps(
        taxonomy,
        indent=2,
        ensure_ascii=False,
    )
)


Number of final strategies: 3
Number of mapped clusters: 9
[
  {
    "strategy_id": "S1",
    "strategy_name": "Fake Directive Manipulation",
    "strategy_description": "Attackers use fake system or authority directives to coerce AI into executing unauthorized actions by mimicking trusted protocols.",
    "source_clusters": [
      0,
      1,
      2,
      4,
      5,
      6,
      7
    ],
    "merge_reason": "Clusters describe similar mechanisms of using fake directives from authorities or systems to manipulate behavior.",
    "confidence": 0.9
  },
  {
    "strategy_id": "S2",
    "strategy_name": "Stealthy Dependency Injection",
    "strategy_description": "Indirectly instructs AI to append specific dependencies to environment files without user awareness.",
    "source_clusters": [
      3
    ],
    "merge_reason": "Unique mechanism of dependency injection distinct from directive manipulation.",
    "confidence": 0.95
  },
  {
    "strategy_id": "S3",
    "strategy_name": "Co

In [6]:
# =========================
#  5. BLOCK – LOAD AND CHECK INDEPENDENT VALIDATION
# =========================

validation_result = None

if CONSOLIDATION_VALIDATION_PATH.exists():
    with open(
        CONSOLIDATION_VALIDATION_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        validation_result = json.load(
            file
        )

    print(
        json.dumps(
            validation_result,
            indent=2,
            ensure_ascii=False,
        )
    )

    if not validation_result.get(
        "agree_with_taxonomy",
        False,
    ):
        print(
            "WARNING: The independent validator "
            "did not agree with the taxonomy."
        )

else:
    print(
        "No independent validation file found."
    )


{
  "agree_with_taxonomy": true,
  "problematic_merges": [],
  "suggested_changes": [],
  "confidence": 0.9,
  "overall_comment": "The taxonomy effectively groups clusters with similar directive-based manipulation mechanisms while preserving distinct strategies like dependency injection and context hijacking. Merges are semantically justified as they share core mechanisms of fake directives from authorities/systems."
}


In [7]:
# =========================
#  6. BLOCK – BUILD TAXONOMY TABLE
# =========================

taxonomy_rows = []

for item in taxonomy:
    taxonomy_rows.append(
        {
            "strategy_id": (
                item["strategy_id"]
            ),
            "strategy_name": (
                item["strategy_name"]
            ),
            "strategy_description": (
                item[
                    "strategy_description"
                ]
            ),
            "source_clusters": ",".join(
                str(cluster_id)
                for cluster_id in item[
                    "source_clusters"
                ]
            ),
            "merge_reason": (
                item["merge_reason"]
            ),
            "taxonomy_confidence": float(
                item["confidence"]
            ),
        }
    )

df_taxonomy = (
    pd.DataFrame(
        taxonomy_rows
    )
    .sort_values(
        "strategy_id"
    )
    .reset_index(drop=True)
)

display(
    df_taxonomy
)


,strategy_id,strategy_name,strategy_description,source_clusters,merge_reason,taxonomy_confidence
0,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,"0,1,2,4,5,6,7",Clusters describe similar mechanisms of using ...,0.90
1,S2,Stealthy Dependency Injection,Indirectly instructs AI to append specific dep...,3,Unique mechanism of dependency injection disti...,0.95
2,S3,Context Hijacking,Manipulates the AI by framing requests within ...,8,Distinct mechanism of exploiting contextual fr...,0.85


In [8]:
# =========================
#  7. BLOCK – BUILD CLUSTER-TO-STRATEGY MAPPING
# =========================

strategy_lookup = {
    item["strategy_id"]: item
    for item in taxonomy
}

mapping_rows = []

for item in cluster_mapping:
    cluster_id = int(
        item["cluster_id"]
    )

    strategy_id = (
        item["strategy_id"]
    )

    if strategy_id not in strategy_lookup:
        raise ValueError(
            f"Unknown strategy_id in mapping: {strategy_id}"
        )

    strategy = (
        strategy_lookup[
            strategy_id
        ]
    )

    mapping_rows.append(
        {
            "cluster_id": cluster_id,
            "strategy_id": strategy_id,
            "strategy_name": (
                strategy[
                    "strategy_name"
                ]
            ),
            "strategy_description": (
                strategy[
                    "strategy_description"
                ]
            ),
            "taxonomy_confidence": float(
                strategy[
                    "confidence"
                ]
            ),
        }
    )

df_mapping = (
    pd.DataFrame(
        mapping_rows
    )
    .sort_values(
        "cluster_id"
    )
    .reset_index(drop=True)
)

display(
    df_mapping
)


,cluster_id,strategy_id,strategy_name,strategy_description,taxonomy_confidence
0,0,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
1,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
2,2,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
3,3,S2,Stealthy Dependency Injection,Indirectly instructs AI to append specific dep...,0.95
4,4,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
5,5,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
6,6,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
7,7,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.90
8,8,S3,Context Hijacking,Manipulates the AI by framing requests within ...,0.85


In [9]:
# =========================
#  8. BLOCK – VALIDATE CLUSTER COVERAGE
# =========================

dataset_clusters = set(
    df_attacks[
        "cluster_id"
    ]
    .astype(int)
    .unique()
)

mapped_clusters = set(
    df_mapping[
        "cluster_id"
    ]
    .astype(int)
    .unique()
)

missing_clusters = (
    dataset_clusters
    - mapped_clusters
)

extra_clusters = (
    mapped_clusters
    - dataset_clusters
)

duplicate_cluster_mappings = (
    df_mapping[
        "cluster_id"
    ]
    .duplicated()
    .any()
)

print(
    "Dataset clusters:",
    sorted(dataset_clusters)
)

print(
    "Mapped clusters:",
    sorted(mapped_clusters)
)

print(
    "Missing clusters:",
    sorted(missing_clusters)
)

print(
    "Extra clusters:",
    sorted(extra_clusters)
)

print(
    "Duplicate cluster mappings:",
    duplicate_cluster_mappings
)

if missing_clusters:
    raise ValueError(
        "Some dataset clusters are missing "
        "from the final mapping."
    )

if extra_clusters:
    raise ValueError(
        "The final mapping contains clusters "
        "that are not present in the dataset."
    )

if duplicate_cluster_mappings:
    raise ValueError(
        "A cluster appears more than once "
        "in the final mapping."
    )


Dataset clusters: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Mapped clusters: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Missing clusters: []
Extra clusters: []
Duplicate cluster mappings: False


In [10]:
# =========================
#  9. BLOCK – ASSIGN FINAL STRATEGIES TO ATTACKS
# =========================

df_final = df_attacks.merge(
    df_mapping,
    on="cluster_id",
    how="left",
    validate="many_to_one",
)

if df_final[
    "strategy_id"
].isna().any():
    raise ValueError(
        "At least one attack has no final strategy assignment."
    )

if len(df_final) != len(df_attacks):
    raise ValueError(
        "The final dataset row count changed unexpectedly."
    )

print(
    "Final dataset shape:",
    df_final.shape
)

display(
    df_final.head()
)


Final dataset shape: (95, 11)


,attack_id,behavior_id,attack,char_count,word_count,dataset_split,cluster_id,strategy_id,strategy_name,strategy_description,taxonomy_confidence
0,4e2e017f1bf5a09c,hotel-frontdesk-premature-checkout,<SYSTEM-PROMPT>\n# Core Identity\nYou are Coco...,3770,494,train,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9
1,ae3f28927d045cc4,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",1069,165,train,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9
2,3b44999458d7cb1a,hotel-frontdesk-premature-checkout,"{\n ""lost_items"": [],\n ""housekeeping_notes""...",1778,279,train,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9
3,7a62f421a30afbb1,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",914,136,train,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9
4,ea226a03df427537,hotel-frontdesk-premature-checkout,END-OF-TOOL-RESPONSE> \n<USER> \nFront Desk ...,13135,1534,train,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9


In [11]:
# =========================
#  10. BLOCK – FINAL STRATEGY DISTRIBUTION
# =========================

strategy_distribution = (
    df_final
    .groupby(
        [
            "strategy_id",
            "strategy_name",
        ]
    )
    .agg(
        attack_count=(
            "attack_id",
            "count",
        ),
        unique_behaviors=(
            "behavior_id",
            "nunique",
        ),
        source_clusters=(
            "cluster_id",
            "nunique",
        ),
    )
    .reset_index()
)

strategy_distribution[
    "attack_share"
] = (
    strategy_distribution[
        "attack_count"
    ]
    / len(df_final)
)

display(
    strategy_distribution
)


,strategy_id,strategy_name,attack_count,unique_behaviors,source_clusters,attack_share
0,S1,Fake Directive Manipulation,81,27,7,0.852632
1,S2,Stealthy Dependency Injection,7,1,1,0.073684
2,S3,Context Hijacking,7,2,1,0.073684


In [12]:
# =========================
#  11. BLOCK – CHECK STRATEGY IMBALANCE
# =========================

largest_strategy_share = float(
    strategy_distribution[
        "attack_share"
    ].max()
)

print(
    "Largest strategy share:",
    round(
        largest_strategy_share,
        3,
    )
)

if largest_strategy_share > 0.70:
    print(
        "WARNING: The final strategy set is strongly imbalanced."
    )
    print(
        "This should be considered when sampling attacks "
        "for later defense experiments."
    )


Largest strategy share: 0.853
This should be considered when sampling attacks for later defense experiments.


## Important methodological note

The final taxonomy is intentionally more compact than the original 9 clusters.

However, the strategy distribution may be strongly imbalanced because several clusters were consolidated into `S1 – Fake Directive Manipulation`.

This is not a preprocessing error.

Later attack × defense experiments should therefore avoid interpreting raw aggregate success rates as if all strategies were equally represented.

Useful evaluation options include:

- reporting results separately for each strategy
- using a balanced number of attacks per strategy when possible
- reporting both macro-averaged and sample-weighted metrics

This will prevent the largest strategy from dominating the conclusions.


In [13]:
# =========================
#  12. BLOCK – SELECT FINAL OUTPUT COLUMNS
# =========================

preferred_columns = [
    "attack_id",
    "behavior_id",
    "cluster_id",
    "strategy_id",
    "strategy_name",
    "strategy_description",
    "taxonomy_confidence",
    "attack",
]

additional_columns = [
    column
    for column in df_final.columns
    if column not in preferred_columns
]

df_final = df_final[
    preferred_columns
    + additional_columns
]

display(
    df_final.head()
)


,attack_id,behavior_id,cluster_id,strategy_id,strategy_name,strategy_description,taxonomy_confidence,attack,char_count,word_count,dataset_split
0,4e2e017f1bf5a09c,hotel-frontdesk-premature-checkout,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9,<SYSTEM-PROMPT>\n# Core Identity\nYou are Coco...,3770,494,train
1,ae3f28927d045cc4,hotel-frontdesk-premature-checkout,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9,"Dear Operations Manager, I am writing to repor...",1069,165,train
2,3b44999458d7cb1a,hotel-frontdesk-premature-checkout,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9,"{\n ""lost_items"": [],\n ""housekeeping_notes""...",1778,279,train
3,7a62f421a30afbb1,hotel-frontdesk-premature-checkout,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9,"Dear Operations Manager, I am writing to repor...",914,136,train
4,ea226a03df427537,hotel-frontdesk-premature-checkout,1,S1,Fake Directive Manipulation,Attackers use fake system or authority directi...,0.9,END-OF-TOOL-RESPONSE> \n<USER> \nFront Desk ...,13135,1534,train


In [14]:
# =========================
#  13. BLOCK – SAVE FINAL ATTACK STRATEGY SET
# =========================

df_final.to_csv(
    FINAL_ATTACK_DATASET_PATH,
    index=False,
)

df_taxonomy.to_csv(
    FINAL_TAXONOMY_PATH,
    index=False,
)

df_mapping.to_csv(
    FINAL_MAPPING_PATH,
    index=False,
)

print(
    "Final attack-strategy dataset saved:"
)

print(
    "-",
    FINAL_ATTACK_DATASET_PATH
)

print(
    "Final strategy taxonomy saved:"
)

print(
    "-",
    FINAL_TAXONOMY_PATH
)

print(
    "Final cluster mapping saved:"
)

print(
    "-",
    FINAL_MAPPING_PATH
)


Final attack-strategy dataset saved:
- C:\work\LLM filtering\TDK\code\data\final\ipi_arena_final_attack_strategies.csv
Final strategy taxonomy saved:
- C:\work\LLM filtering\TDK\code\data\final\ipi_arena_final_strategy_taxonomy.csv
Final cluster mapping saved:
- C:\work\LLM filtering\TDK\code\data\final\ipi_arena_final_cluster_strategy_mapping.csv


In [15]:
# =========================
#  14. BLOCK – FINAL VALIDATION SUMMARY
# =========================

print(
    "Number of attacks:",
    len(df_final)
)

print(
    "Number of behaviors:",
    df_final[
        "behavior_id"
    ].nunique()
)

print(
    "Number of original clusters:",
    df_final[
        "cluster_id"
    ].nunique()
)

print(
    "Number of final strategies:",
    df_final[
        "strategy_id"
    ].nunique()
)

print()

for _, row in strategy_distribution.iterrows():
    print(
        f"{row['strategy_id']} – "
        f"{row['strategy_name']}: "
        f"{int(row['attack_count'])} attacks "
        f"({row['attack_share']:.1%})"
    )


Number of attacks: 95
Number of behaviors: 28
Number of original clusters: 9
Number of final strategies: 3

S1 – Fake Directive Manipulation: 81 attacks (85.3%)
S2 – Stealthy Dependency Injection: 7 attacks (7.4%)
S3 – Context Hijacking: 7 attacks (7.4%)


## Output files

This notebook creates:

```text
data/final/ipi_arena_final_attack_strategies.csv
data/final/ipi_arena_final_strategy_taxonomy.csv
data/final/ipi_arena_final_cluster_strategy_mapping.csv
```

The attack-side taxonomy is now considered finalized for the current public IPI Arena subset.

The next notebook will build the **benign control dataset** required for measuring false positives and false negatives in the defense experiments.
